# E2 Qwen3-8B QLoRA private Kaggle runner
Attach `e2_kaggle_source.zip` as a private Kaggle dataset and select a GPU runtime. This notebook keeps the experiment configuration unchanged.

In [ ]:
from pathlib import Path
import os, shutil

archives = list(Path('/kaggle/input').rglob('e2_kaggle_source.zip'))
if len(archives) != 1:
    raise RuntimeError(f'Expected one attached e2_kaggle_source.zip, found {archives}')
REPO_DIR = Path('/kaggle/working/nlp-proj')
REPO_DIR.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(str(archives[0]), str(REPO_DIR))
os.chdir(REPO_DIR)
os.environ['PYTHONPATH'] = str(REPO_DIR / 'code')
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print(REPO_DIR)


In [ ]:
!python -m pip install --no-cache-dir -r requirements-llm.txt
import torch
assert torch.cuda.is_available(), 'Select a Kaggle GPU runtime before running E2'
props = torch.cuda.get_device_properties(0)
assert props.total_memory >= 14 * 1024**3, f'E2 requires at least 14 GiB VRAM, found {props.total_memory / 1024**3:.1f}'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), f'{props.total_memory / 1024**3:.1f} GiB')


In [ ]:
!python -m pytest -q tests
!python -m mt_pipeline train --config configs/e2_qwen3_qlora.yaml


Validation is generated and scored before selection is frozen. Test generation is refused by the pipeline until the freeze artifact exists.

In [ ]:
!python -m mt_pipeline predict --config configs/e2_qwen3_qlora.yaml --split val
!python -m mt_pipeline evaluate --predictions predictions/e2_qwen3_8b_qlora_vi_zh_v1.val.jsonl --output metrics/e2_qwen3_8b_qlora_vi_zh_v1.val.json
!python -m mt_pipeline freeze-selection --config configs/e2_qwen3_qlora.yaml --validation-predictions predictions/e2_qwen3_8b_qlora_vi_zh_v1.val.jsonl --validation-metrics metrics/e2_qwen3_8b_qlora_vi_zh_v1.val.json


In [ ]:
!python -m mt_pipeline predict --config configs/e2_qwen3_qlora.yaml --split test
!python -m mt_pipeline evaluate --predictions predictions/e2_qwen3_8b_qlora_vi_zh_v1.test.jsonl --output metrics/e2_qwen3_8b_qlora_vi_zh_v1.test.json
!python -m mt_pipeline project-status


In [ ]:
result_paths = [
    REPO_DIR / 'checkpoint/e2_qwen3_8b_qlora_vi_zh_v1',
    REPO_DIR / 'work/e2_qwen3_8b_qlora_vi_zh_v1',
    REPO_DIR / 'predictions/e2_qwen3_8b_qlora_vi_zh_v1.val.jsonl',
    REPO_DIR / 'predictions/e2_qwen3_8b_qlora_vi_zh_v1.test.jsonl',
    REPO_DIR / 'metrics/e2_qwen3_8b_qlora_vi_zh_v1.val.json',
    REPO_DIR / 'metrics/e2_qwen3_8b_qlora_vi_zh_v1.test.json',
]
missing = [str(path) for path in result_paths if not path.exists()]
assert not missing, missing
archive = shutil.make_archive('/kaggle/working/e2_qwen3_qlora_results', 'zip', root_dir=REPO_DIR,
    base_dir='.')
print(f'Download {archive}; it contains the complete repo including E2 artifacts.')
